# 3. Model Training & Evaluation

Goal: train multiple models, compare them with cross-validation, and save the best one.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from xgboost import XGBRegressor

from src.evaluate import rmse, print_scores, plot_predicted_vs_actual, plot_feature_importance
from src.features import inverse_log_transform

X_train = pd.read_csv('../data/processed/X_train.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()

print('X_train:', X_train.shape)
print('y_train:', y_train.shape)

## Cross-validation comparison

In [ ]:
models = {
    'Ridge Regression': Ridge(alpha=10),
    'Random Forest':    RandomForestRegressor(
                            n_estimators=300, random_state=42, n_jobs=-1),
    'XGBoost':          XGBRegressor(
                            n_estimators=500, learning_rate=0.05,
                            max_depth=4, random_state=42, n_jobs=-1),
}

cv_results = {}
for name, model in models.items():
    scores = cross_val_score(
        model, X_train, y_train,
        scoring='neg_root_mean_squared_error', cv=5
    )
    cv_rmse = -scores.mean()
    cv_results[name] = cv_rmse
    print(f'{name:20s}  CV RMSE (log-scale): {cv_rmse:.4f}  ± {scores.std():.4f}')

## Train best model on full training set

In [ ]:
best_name = min(cv_results, key=cv_results.get)
print(f'Best model: {best_name}  (CV RMSE: {cv_results[best_name]:.4f})')

best_model = models[best_name]
best_model.fit(X_train, y_train)

## Evaluation on training set (diagnostic)

In [ ]:
y_pred_log = best_model.predict(X_train)
y_pred     = inverse_log_transform(pd.Series(y_pred_log))
y_actual   = inverse_log_transform(y_train)

print_scores(best_name, y_actual, y_pred)

plot_predicted_vs_actual(
    y_actual, y_pred, best_name,
    save_path='../reports/figures/predicted_vs_actual.png'
)

## Feature importance

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    plot_feature_importance(
        best_model, X_train.columns, top_n=20,
        save_path='../reports/figures/feature_importance.png'
    )

## Generate Kaggle submission

In [ ]:
X_test = pd.read_csv('../data/processed/X_test.csv')
test_ids = pd.read_csv('../data/raw/test.csv')['Id']

preds_log = best_model.predict(X_test)
preds = inverse_log_transform(pd.Series(preds_log))

submission = pd.DataFrame({'Id': test_ids, 'SalePrice': preds})
submission.to_csv('../data/processed/submission.csv', index=False)
print('Submission saved!')
submission.head()

## Save model

In [ ]:
filename = best_name.lower().replace(' ', '_') + '.pkl'
joblib.dump(best_model, f'../models/{filename}')
print(f'Model saved to models/{filename}')